In [1]:
# ── Cell 2: Explore New Dataset Folder Structure ──────────────────────────────
import os

# 🔧 Set this to your new dataset root path
DATASET_ROOT = r"C:\Users\vkr30\Downloads\Plant Disease\dataset"   # ← update this

def explore_dataset(root, max_depth=3, max_files=5):
    """Print folder tree and class-level image counts."""
    print(f"\n📁 Dataset Root: {root}\n")
    print("=" * 60)

    all_classes   = []
    total_images  = 0
    split_summary = {}

    for depth_0 in sorted(os.listdir(root)):               # train / val / test
        d0 = os.path.join(root, depth_0)
        if not os.path.isdir(d0):
            continue
        print(f"\n📂 {depth_0}/")
        split_summary[depth_0] = {"classes": 0, "images": 0}

        for depth_1 in sorted(os.listdir(d0)):             # class folders
            d1 = os.path.join(d0, depth_1)
            if not os.path.isdir(d1):
                continue
            imgs = [f for f in os.listdir(d1)
                    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.webp'))]
            print(f"   └── {depth_1:45s} → {len(imgs):>5} images")
            split_summary[depth_0]["classes"] += 1
            split_summary[depth_0]["images"]  += len(imgs)
            total_images += len(imgs)
            if depth_1 not in all_classes:
                all_classes.append(depth_1)

    print("\n" + "=" * 60)
    print("📊 Summary:")
    for split, info in split_summary.items():
        print(f"   {split:10s}: {info['classes']} classes, {info['images']} images")
    print(f"\n   Total classes : {len(all_classes)}")
    print(f"   Total images  : {total_images}")
    print("\n📋 Class names detected:")
    for i, cls in enumerate(sorted(all_classes)):
        print(f"   [{i:>2}] {cls}")

    return sorted(all_classes)

class_names = explore_dataset(DATASET_ROOT)


📁 Dataset Root: C:\Users\vkr30\Downloads\Plant Disease\dataset


📂 test/
   └── Apple___Apple_scab                            →   504 images
   └── Apple___Black_rot                             →   497 images
   └── Apple___Cedar_apple_rust                      →   440 images
   └── Apple___healthy                               →   502 images
   └── Blueberry___healthy                           →   454 images
   └── Cherry_(including_sour)___Powdery_mildew      →   421 images
   └── Cherry_(including_sour)___healthy             →   456 images
   └── Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot →   410 images
   └── Corn_(maize)___Common_rust_                   →   477 images
   └── Corn_(maize)___Northern_Leaf_Blight           →   477 images
   └── Corn_(maize)___healthy                        →   465 images
   └── Grape___Black_rot                             →   472 images
   └── Grape___Esca_(Black_Measles)                  →   480 images
   └── Grape___Leaf_blight_(Isariopsi

In [2]:
# ── Cell 3: Generate dataset YAML ─────────────────────────────────────────────
import yaml, os

DATASET_ROOT = r"C:\Users\vkr30\Downloads\Plant Disease\dataset"  # ← update this

class_names = [
    "Apple___Apple_scab",
    "Apple___Black_rot",
    "Apple___Cedar_apple_rust",
    "Apple___healthy",
    "Blueberry___healthy",
    "Cherry_(including_sour)___Powdery_mildew",
    "Cherry_(including_sour)___healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Grape___Black_rot",
    "Grape___Esca_(Black_Measles)",
    "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)",
    "Grape___healthy",
    "Orange___Haunglongbing_(Citrus_greening)",
    "Peach___Bacterial_spot",
    "Peach___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Raspberry___healthy",
    "Soybean___healthy",
    "Squash___Powdery_mildew",
    "Strawberry___Leaf_scorch",
    "Strawberry___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]

yaml_data = {
    "path"  : DATASET_ROOT,
    "train" : "train",
    "val"   : "test",     # using test folder as val (no separate val split)
    "nc"    : len(class_names),
    "names" : class_names,
}

yaml_path = os.path.join(DATASET_ROOT, "plant_disease.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"✅ YAML saved to: {yaml_path}")
print(f"   Classes : {len(class_names)}")
print(f"   Train   : {os.path.join(DATASET_ROOT, 'train')}")
print(f"   Val     : {os.path.join(DATASET_ROOT, 'test')}")

✅ YAML saved to: C:\Users\vkr30\Downloads\Plant Disease\dataset\plant_disease.yaml
   Classes : 38
   Train   : C:\Users\vkr30\Downloads\Plant Disease\dataset\train
   Val     : C:\Users\vkr30\Downloads\Plant Disease\dataset\test


In [3]:
# ── Cell 5: Train YOLO11m Classification ─────────────────────────────────────
from ultralytics import YOLO

# ✅ For classification: pass the DATASET FOLDER, not the yaml file
DATASET_ROOT = r"C:\Users\vkr30\Downloads\Plant Disease\dataset"

# YOLO11m — best fit for RTX 4060 8GB with 38 classes
model = YOLO("yolo11m-cls.pt")   # auto-downloads pretrained weights

model.train(
    data         = DATASET_ROOT,   # ← directory, not yaml
    epochs       = 100,
    imgsz        = 224,
    device       = 0,              # RTX 4060
    batch        = 32,             # drop to 16 if OOM
    workers      = 4,
    optimizer    = "AdamW",
    lr0          = 0.001,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    patience     = 20,             # early stopping
    verbose      = False,
    project      = "plant_disease_cls",
    name         = "yolo11m_38cls",
    save         = True,
    plots        = True,
)

print("\n✅ Training complete!")
print("📍 Best weights : plant_disease_cls/yolo11m_38cls/weights/best.pt")

Ultralytics 8.4.37  Python-3.10.20 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\vkr30\Downloads\Plant Disease\dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11m_38cls2, nbs=64, nms=False, opset=None, optimize=Fal

In [4]:
# ── Cell 6: Evaluate ──────────────────────────────────────────────────────────
from ultralytics import YOLO

best_weights = r"C:\Users\vkr30\runs\classify\plant_disease_cls\yolo11m_38cls2\weights\best.pt"
model = YOLO(best_weights)

metrics = model.val(verbose=False)

print("\n📊 Final Evaluation Metrics:")
print(f"   🔹 Top-1 Accuracy : {metrics.top1 * 100:.2f}%")
print(f"   🔹 Top-5 Accuracy : {metrics.top5 * 100:.2f}%")

Ultralytics 8.4.37  Python-3.10.20 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO11m-cls summary (fused): 57 layers, 10,390,310 parameters, 0 gradients, 39.3 GFLOPs
WARNING Dataset 'split=val' not found, using 'split=test' instead.
train: C:\Users\vkr30\Downloads\Plant Disease\dataset\train... found 70295 images in 38 classes  
val: C:\Users\vkr30\Downloads\Plant Disease\dataset\test... found 17572 images in 38 classes  
test: C:\Users\vkr30\Downloads\Plant Disease\dataset\test... found 17572 images in 38 classes  
val: Fast image access  (ping: 0.10.0 ms, read: 64.714.7 MB/s, size: 12.1 KB)
val: Scanning C:\Users\vkr30\Downloads\Plant Disease\dataset\test... 17572 images, 0 corrupt: 100% ━━━━━━━━━━━━ 17572/17572  0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1099/1099 61.8it/s 17.8s<0.0s
                   all          1          1
Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\U